In [1]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.3 MB/s eta 0:00:00


In [14]:
import pandas as pd
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader  # Updated import
from sklearn.preprocessing import StandardScaler
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

In [3]:
# Load the dataset
df = pd.read_csv('/content/EVSE-B-PowerCombined.csv')

# Drop irrelevant columns if needed
df = df[['time', 'shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW', 'State', 'Attack', 'Label']]
df

,time,shunt_voltage,bus_voltage_V,current_mA,power_mW,State,Attack,Label
0,12/25/2023 22:35,978,5.165,1027,5300,idle,syn-flood,attack
1,12/25/2023 22:35,872,5.161,1009,4980,idle,syn-flood,attack
2,12/25/2023 22:35,1017,5.165,1029,5300,idle,syn-flood,attack
3,12/25/2023 22:35,930,5.161,1005,5180,idle,syn-flood,attack
4,12/25/2023 22:35,958,5.165,1034,5180,idle,syn-flood,attack
...,...,...,...,...,...,...,...,...
115293,12/30/2023 11:51,484,5.201,487,2500,idle,Backdoor,attack
115294,12/30/2023 11:51,477,5.197,567,2860,idle,Backdoor,attack
115295,12/30/2023 11:51,510,5.197,509,2660,idle,Backdoor,attack
115296,12/30/2023 11:51,477,5.201,477,2500,idle,Backdoor,attack


In [4]:
# Convert labels to binary (0 for normal, 1 for attack)
df['Label'] = df['Label'].apply(lambda x: 1 if x == 'attack' else 0)
df

,time,shunt_voltage,bus_voltage_V,current_mA,power_mW,State,Attack,Label
0,12/25/2023 22:35,978,5.165,1027,5300,idle,syn-flood,1
1,12/25/2023 22:35,872,5.161,1009,4980,idle,syn-flood,1
2,12/25/2023 22:35,1017,5.165,1029,5300,idle,syn-flood,1
3,12/25/2023 22:35,930,5.161,1005,5180,idle,syn-flood,1
4,12/25/2023 22:35,958,5.165,1034,5180,idle,syn-flood,1
...,...,...,...,...,...,...,...,...
115293,12/30/2023 11:51,484,5.201,487,2500,idle,Backdoor,1
115294,12/30/2023 11:51,477,5.197,567,2860,idle,Backdoor,1
115295,12/30/2023 11:51,510,5.197,509,2660,idle,Backdoor,1
115296,12/30/2023 11:51,477,5.201,477,2500,idle,Backdoor,1


In [5]:
# Standardize the features
scaler = StandardScaler()
features = scaler.fit_transform(df[['shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW']])
features

array([[ 1.81660652, -1.6022681 ,  2.06393495,  2.06335774],
       [ 1.27904704, -1.87847554,  1.97270891,  1.74701557],
       [ 2.01438783, -1.6022681 ,  2.07407118,  2.06335774],
       ...,
       [-0.55676927,  0.60739141, -0.56134761, -0.54646521],
       [-0.7241227 ,  0.88359885, -0.72352723, -0.7046363 ],
       [ 0.05685994,  0.05497653,  0.00628105,  0.02690498]])

In [6]:
# Create edge index (assuming each row is a node connected to the next for simplicity)
edge_index = torch.tensor([[i, i + 1] for i in range(len(df) - 1)], dtype=torch.long).t()

# Convert to PyTorch Geometric data format
x = torch.tensor(features, dtype=torch.float)
y = torch.tensor(df['Label'].values, dtype=torch.long)
data = Data(x=x, edge_index=edge_index, y=y)
data

Data(x=[115298, 4], edge_index=[2, 115297], y=[115298])

In [7]:
# Train-validation-test split
train_data, temp_data = train_test_split(list(range(len(df))), test_size=0.4, stratify=y)
val_data, test_data = train_test_split(temp_data, test_size=0.5, stratify=y[temp_data])

train_mask = torch.tensor([i in train_data for i in range(len(df))], dtype=torch.bool)
val_mask = torch.tensor([i in val_data for i in range(len(df))], dtype=torch.bool)
test_mask = torch.tensor([i in test_data for i in range(len(df))], dtype=torch.bool)

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

train_loader = DataLoader([data], batch_size=1, shuffle=True)
val_loader = DataLoader([data], batch_size=1)
test_loader = DataLoader([data], batch_size=1)

train_loader
val_loader
test_loader

In [15]:
class EVChargingGNN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(EVChargingGNN, self).__init__()
        # Replace GCNConv with GATConv
        self.conv1 = GATConv(input_dim, hidden_dim, heads=4, concat=True)  # Multi-head attention
        self.conv2 = GATConv(hidden_dim * 4, hidden_dim, heads=4, concat=True)
        self.fc = torch.nn.Linear(hidden_dim * 4, output_dim)  # Adjust for concatenation of heads
        self.dropout = 0.3

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.fc(x)
        return torch.sigmoid(x).squeeze()

# Instantiate the model
model = EVChargingGNN(input_dim=x.shape[1], hidden_dim=16, output_dim=1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = torch.nn.BCELoss()

In [16]:
def train():
    model.train()
    total_loss = 0
    for data in train_loader:
        optimizer.zero_grad()
        output = model(data)[data.train_mask]
        loss = criterion(output, data.y[data.train_mask].float())
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def evaluate(loader, mask_type):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for data in loader:
            output = model(data)[getattr(data, f"{mask_type}_mask")]
            preds = (output > 0.5).float()  # Threshold at 0.5 for binary classification
            all_preds.append(preds.cpu())
            all_labels.append(data.y[getattr(data, f"{mask_type}_mask")].cpu())
            all_probs.append(output.cpu())

    # Concatenate predictions, labels, and probabilities
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    all_probs = torch.cat(all_probs).numpy()

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    roc_auc = None
    if len(set(all_labels)) > 1:
        roc_auc = roc_auc_score(all_labels, all_probs)

    return accuracy, precision, f1, roc_auc

# Training loop with validation
for epoch in range(50):
    loss = train()
    val_accuracy, val_precision, val_f1, val_auc = evaluate(val_loader, 'val')
    print(f'Epoch {epoch+1}, Loss: {loss:.4f}, Val Accuracy: {val_accuracy:.4f}, Val Precision: {val_precision:.4f}, Val F1: {val_f1:.4f}, Val AUC: {val_auc:.4f}')

# Test evaluation
test_accuracy, test_precision, test_f1, test_roc_auc = evaluate(test_loader, 'test')
print(f'Test Accuracy: {test_accuracy:.4f}')
print(f'Test Precision: {test_precision:.4f}')
print(f'Test F1 Score: {test_f1:.4f}')
print(f'Test ROC-AUC: {test_roc_auc:.4f}')

Epoch 1, Loss: 0.6588, Val Accuracy: 0.8755, Val Precision: 0.8755, Val F1: 0.9336, Val AUC: 0.6790
Epoch 2, Loss: 0.5663, Val Accuracy: 0.8755, Val Precision: 0.8755, Val F1: 0.9336, Val AUC: 0.6837
Epoch 3, Loss: 0.4842, Val Accuracy: 0.8755, Val Precision: 0.8755, Val F1: 0.9336, Val AUC: 0.6848
Epoch 4, Loss: 0.4134, Val Accuracy: 0.8755, Val Precision: 0.8755, Val F1: 0.9336, Val AUC: 0.6855
Epoch 5, Loss: 0.3633, Val Accuracy: 0.8755, Val Precision: 0.8755, Val F1: 0.9336, Val AUC: 0.6861
Epoch 6, Loss: 0.3417, Val Accuracy: 0.8755, Val Precision: 0.8755, Val F1: 0.9336, Val AUC: 0.6881
Epoch 7, Loss: 0.3489, Val Accuracy: 0.8755, Val Precision: 0.8755, Val F1: 0.9336, Val AUC: 0.6926
Epoch 8, Loss: 0.3640, Val Accuracy: 0.8755, Val Precision: 0.8755, Val F1: 0.9336, Val AUC: 0.6981
Epoch 9, Loss: 0.3698, Val Accuracy: 0.8755, Val Precision: 0.8755, Val F1: 0.9336, Val AUC: 0.7008
Epoch 10, Loss: 0.3668, Val Accuracy: 0.8755, Val Precision: 0.8755, Val F1: 0.9336, Val AUC: 0.7014